In [8]:
from sklearn.datasets import load_wine
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.neighbors import KNeighborsClassifier
from sklearn.feature_selection import SequentialFeatureSelector

"""
Wine dataset
↓
train/test split
↓
StandardScaler
↓
KNN
↓
SequentialFeatureSelector
↓
select features using TRAINING data
↓
transform train/test
↓
train KNN again
↓
compare accuracy
"""

wine = load_wine()

X = wine.data
y = wine.target

print(X.shape)
print(wine.feature_names)

# Split the dataset into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.3,
    random_state=1,
    stratify=y
)

# 因为今天用的是 KNN，所以先 scaling
sc = StandardScaler()
X_train_std = sc.fit_transform(X_train)
X_test_std = sc.transform(X_test)


# 建立 baseline (13 features)
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_std, y_train)

train_acc_all = knn.score(X_train_std, y_train)
test_acc_all = knn.score(X_test_std, y_test)

print("All features:")
print("Train accuracy:", train_acc_all)
print("Test accuracy:", test_acc_all)


"""
knn → 用哪个 model 判断 feature subset 好不好

n_features_to_select=5 → 最后留下 5 个 features

direction="backward" → 从全部 features 开始，逐渐删除 features

scoring="accuracy" → 用 accuracy 评价 candidate subsets

cv=5 → 用 5-fold cross-validation 评价

"""
sfs = SequentialFeatureSelector(
    knn,
    n_features_to_select=5,
    direction="backward",
    scoring="accuracy",
    cv=5
)

sfs.fit(X_train_std, y_train)

# True → selected， False → removed
selected_mask = sfs.get_support()
print(selected_mask)

selected_features = [
    feature
    for feature, selected in zip(wine.feature_names, selected_mask)
    if selected
]

print("Selected features:")
for feature in selected_features:
    print(feature)

# transform train/test
X_train_selected = sfs.transform(X_train_std)
X_test_selected = sfs.transform(X_test_std)

print(X_train_std.shape)
print(X_train_selected.shape)
print(X_test_std.shape)
print(X_test_selected.shape)

# 重新训练 KNN

knn_selected = KNeighborsClassifier(n_neighbors=5)

knn_selected.fit(X_train_selected, y_train)

train_acc_selected = knn_selected.score(
    X_train_selected,
    y_train
)

test_acc_selected = knn_selected.score(
    X_test_selected,
    y_test
)

print("All features")
print("Train:", train_acc_all)
print("Test:", test_acc_all)

print()

print("Selected features")
print("Train:", train_acc_selected)
print("Test:", test_acc_selected)


(178, 13)
['alcohol', 'malic_acid', 'ash', 'alcalinity_of_ash', 'magnesium', 'total_phenols', 'flavanoids', 'nonflavanoid_phenols', 'proanthocyanins', 'color_intensity', 'hue', 'od280/od315_of_diluted_wines', 'proline']
All features:
Train accuracy: 0.9596774193548387
Test accuracy: 0.9259259259259259
[ True  True False False False False False False False False  True  True
  True]
Selected features:
alcohol
malic_acid
hue
od280/od315_of_diluted_wines
proline
(124, 13)
(124, 5)
(54, 13)
(54, 5)
All features
Train: 0.9596774193548387
Test: 0.9259259259259259

Selected features
Train: 0.9758064516129032
Test: 0.9814814814814815
